# Huấn luyện QLoRA với checkpoint trên Google Drive

Chọn GPU T4 trong **Runtime → Change runtime type**. Đặt gói nguồn `qlora_colab_source.zip` vào `MyDrive/vietnamese-legal-qlora/` trước khi chạy. Gói giữ nguyên dữ liệu đã kiểm chứng và kết quả baseline. Notebook ghi checkpoint vào Drive. Các lần chạy sau dùng cùng `RUN_NAME` để tiếp tục.

Qwen2.5-3B-Instruct là Transformer decoder-only. Lượng tử hóa NF4 nén các lớp tuyến tính nền xuống 4 bit. Double quantization tiếp tục nén các hằng số lượng tử hóa. LoRA giữ trọng số nền và học hai ma trận hạng thấp, với rank 8, alpha 16 và dropout 0,05. QLoRA lan truyền gradient qua mô hình nền lượng tử hóa đến adapter. SFT dùng cross-entropy trên token assistant và token kết thúc lượt. Batch 1, tích lũy 8 bước, learning rate 2e-4, chiều dài tối đa 1.024 và seed 42. Run đã khởi tạo dùng 2 epoch để giới hạn chi phí tính toán và giữ nguyên lịch optimizer khi resume.

In [ ]:
from google.colab import drive
from pathlib import Path
import subprocess
import sys
from zipfile import ZipFile

drive.mount("/content/drive")
DRIVE = Path("/content/drive/MyDrive/vietnamese-legal-qlora")
SOURCE = DRIVE / "qlora_colab_source.zip"
ROOT = Path("/content/vietnamese-legal-qlora")
RUN_NAME = "qwen3b-r8-local"
RANK = 8
RUN = DRIVE / "runs" / RUN_NAME
if not SOURCE.is_file():
    raise FileNotFoundError(f"Tải gói nguồn lên {SOURCE}")
with ZipFile(SOURCE) as archive:
    for member in archive.infolist():
        target = (ROOT / member.filename).resolve()
        if not target.is_relative_to(ROOT) or member.is_dir():
            raise ValueError(f"Tệp nguồn không hợp lệ: {member.filename}")
        payload = archive.read(member)
        if target.exists() and target.read_bytes() != payload:
            raise ValueError(f"Mã nguồn đã thay đổi tại {target}. Mở runtime mới.")
        target.parent.mkdir(parents=True, exist_ok=True)
        if not target.exists():
            target.write_bytes(payload)
subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv"], check=True)
if not (RUN / "run_config.json").exists():
    raise FileNotFoundError(f"Chép thư mục run đã giải nén vào {RUN} trước khi chạy.")

## Môi trường và kiểm tra dữ liệu

Dùng Python 3.12 hoặc 3.13 và các phiên bản thư viện đã ghim. Kiểm tra đủ 400 mẫu train, 50 validation và cách tính loss trên token của assistant.

In [ ]:
if sys.version_info[:2] not in {(3, 12), (3, 13)}:
    raise RuntimeError("Chọn runtime Python 3.12 hoặc 3.13.")
VENV = ROOT / ".venv"
PY = VENV / "bin/python"
if not PY.exists():
    subprocess.run([sys.executable, "-m", "venv", "--without-pip", str(VENV)], check=True)
if subprocess.run([str(PY), "-m", "pip", "--version"], capture_output=True).returncode:
    subprocess.run([sys.executable, "-m", "pip", "--python", str(PY), "install", "pip"], check=True)
for requirements in ("requirements-torch.txt", "requirements.txt", "requirements-analysis.txt"):
    subprocess.run([str(PY), "-m", "pip", "install", "-r", requirements], cwd=ROOT, check=True)
train = [str(PY), "-X", "utf8", "scripts/train_qlora.py", "--run-dir", str(RUN), "--rank", str(RANK)]
subprocess.run(train + ["--check-only"], cwd=ROOT, check=True)
subprocess.run(train + ["--status"], cwd=ROOT, check=True)

## Khôi phục khóa sau khi Colab bị ngắt

Chỉ bật `RECOVER_STALE_LOCK` khi chắc chắn runtime cũ đã dừng. Cell đổi tên khóa cũ và giữ lại thông tin chủ sở hữu. Hãy để giá trị mặc định nếu lần chạy trước kết thúc bình thường.

In [ ]:
from uuid import uuid4

RECOVER_STALE_LOCK = False
lock = RUN / ".writer"
if lock.exists():
    owner = lock / "owner.json"
    print(owner.read_text(encoding="utf-8") if owner.exists() else "Chưa ghi được thông tin chủ sở hữu.")
    if RECOVER_STALE_LOCK:
        lock.rename(RUN / f".writer-stale-{uuid4().hex}")
    else:
        raise RuntimeError("Xác nhận runtime cũ đã dừng trước khi mở khóa.")

## Huấn luyện hoặc tiếp tục

Mặc định chạy hai epoch. Lưu checkpoint mỗi 5 bước cập nhật. Giữ ba checkpoint hoạt động, chuyển các bản cũ vào `history/` trên Drive. Đặt `PAUSE_AT_STEP` nếu muốn kết thúc sớm ở một bước mới và lưu checkpoint trước khi thoát.

In [ ]:
PAUSE_AT_STEP = None
pause = [] if PAUSE_AT_STEP is None else ["--pause-at-step", str(PAUSE_AT_STEP)]
subprocess.run(train + pause, cwd=ROOT, check=True)
subprocess.run(train + ["--status"], cwd=ROOT, check=True)

## Kết quả huấn luyện

Cell này đọc kết quả đã sinh trên Drive sau khi có adapter cuối và chạy kiểm thử nạp lại.

In [ ]:
from IPython.display import Image, display
import json

adapter = RUN / "final_adapter"
if adapter.is_dir():
    for script, options in (("summarize_training.py", []), ("demo.py", ["--check"])):
        subprocess.run([str(PY), "-X", "utf8", f"scripts/{script}", "--adapter", str(adapter), *options], cwd=ROOT, check=True)
    print(json.dumps(json.loads((adapter / "training_metrics.json").read_text(encoding="utf-8")), indent=2))
    display(Image(filename=str(RUN / "training_summary/training_curve.png")))
else:
    print("Chưa có adapter cuối. Kiểm tra checkpoint đã lưu ở cell trước.")

## Đánh giá adapter trên tập test đã ghim

Chỉ chạy sau khi `final_adapter` có bản ghi hoàn tất. Kết quả và từng dự đoán nằm trên Drive. Baseline đo trên RTX 3050 Ti Laptop nên độ trễ của Colab được báo riêng.

In [ ]:
RUN_EVALUATION = False
if RUN_EVALUATION:
    adapter = RUN / "final_adapter"
    evaluation = DRIVE / "evaluations" / RUN_NAME
    subprocess.run(
        [str(PY), "-X", "utf8", "scripts/evaluate_adapter.py", "--adapter", str(adapter), "--run-dir", str(evaluation)],
        cwd=ROOT, check=True,
    )

## Demo Gradio

Bật cell khi đã có adapter cuối. Chạy bằng Python của notebook nếu môi trường hiện tại đã có các gói đã ghim. Nếu phiên Colab yêu cầu chia sẻ công khai, dùng hướng dẫn local.

In [ ]:
RUN_INLINE_DEMO = False
if RUN_INLINE_DEMO:
    for requirements in ("requirements-torch.txt", "requirements.txt", "requirements-analysis.txt"):
        subprocess.run([sys.executable, "-m", "pip", "install", "-r", requirements], cwd=ROOT, check=True)
    import sys
    sys.path.insert(0, str(ROOT / "app"))
    from app import build_app
    build_app(RUN / "final_adapter").queue().launch(inline=True, share=False)